# Ретроспективная проверка Hutton и Полякова на VAAD

Единица анализа — `field_uid × season`. Первое зарегистрированное появление болезни задаётся интервалом `(последний отрицательный визит, первый положительный визит]`. Основной срез использует только прямые геопривязки; восстановленные однозначные привязки добавляются как анализ чувствительности.

Период 2015–2022 используется как development, 2023–2025 — как temporal test slice (временной тестовый срез), а не untouched external holdout: правила анализа уточнялись после знакомства с набором данных. Сезон 2026 исключён как незавершённый. Отрицательные сезоны используются только как анализ чувствительности, а не как оценка specificity: без стандартизированного протокола осмотра, полного наблюдения сезона и данных об обработках они не считаются доказанными true negative.

In [ ]:
from pathlib import Path
import os, subprocess, sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    REPO_DIR = Path('/content/AgroPhenology')
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', 'https://github.com/vkonov2/AgroPhenology.git', str(REPO_DIR)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
else:
    REPO_DIR = Path.cwd()
    if not (REPO_DIR / 'pyproject.toml').exists():
        REPO_DIR = Path.cwd().parent

print('Repository:', REPO_DIR.resolve())

In [ ]:
# В Colab файл VAAD не лежит в Git, поэтому по умолчанию откроется загрузка файла.
INPUT_MODE = 'upload' if IN_COLAB else 'local'  # upload | drive | local
LOCAL_CSV_PATH = REPO_DIR / 'data/vaad_observations_geocoded_recovered.csv'
DRIVE_CSV_PATH = '/content/drive/MyDrive/vaad_observations_geocoded_recovered.csv'
MAXIMUM_COMPLETE_SEASON = 2025
BATCH_SIZE = 20
PERMUTATIONS = 20000
OUTPUT_DIR = REPO_DIR / 'results/vaad_late_blight'
CACHE_DIR = REPO_DIR / 'data/cache/vaad_open_meteo'

In [ ]:
if INPUT_MODE == 'upload':
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Загрузите ровно один CSV')
    CSV_PATH = Path('/content') / next(iter(uploaded))
elif INPUT_MODE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    CSV_PATH = Path(DRIVE_CSV_PATH)
elif INPUT_MODE == 'local':
    CSV_PATH = Path(LOCAL_CSV_PATH)
else:
    raise ValueError(f'Неизвестный INPUT_MODE: {INPUT_MODE}')

if not CSV_PATH.exists():
    raise FileNotFoundError(CSV_PATH)
print('Input:', CSV_PATH.resolve(), f'({CSV_PATH.stat().st_size / 1024**2:.1f} MB)')

In [ ]:
from agro_phenology.vaad_validation import run_vaad_validation

summary = run_vaad_validation(
    CSV_PATH,
    output_dir=OUTPUT_DIR,
    cache_dir=CACHE_DIR,
    maximum_season=MAXIMUM_COMPLETE_SEASON,
    batch_size=BATCH_SIZE,
    repetitions=PERMUTATIONS,
)
print('SHA-256:', summary['input']['sha256'])

In [ ]:
import pandas as pd

COHORT = 'holdout_2023_2025_unseen_fields'
hutton_rows = []
calendar_rows = []
polyakov_rows = []
for mode in ('direct', 'expanded'):
    cohort = summary['hutton'][mode][COHORT]
    for lookback in (7, 14, 21):
        metric = cohort[f'lookback_{lookback}d']
        observed = metric['first_detection']
        null = metric['within_season_date_permutation_null']
        burden_june1 = metric['pre_detection_alarm_burden_june1']
        burden_first_visit = metric['pre_detection_alarm_burden_first_visit_proxy']
        hutton_rows.append({
            'mode': mode,
            'lookback_days': lookback,
            'n': observed['n'],
            'hits': observed['successes'],
            'hit_rate': observed['rate'],
            'within_season_null': null['null_mean_rate'],
            'lift_vs_null': null['lift'],
            'permutation_p_one_sided': null['permutation_p_one_sided'],
            'pre_detection_alarm_fraction_june1': burden_june1['alarm_day_fraction'],
            'pre_detection_alarm_fraction_first_visit_proxy': burden_first_visit['alarm_day_fraction'],
            'pre_detection_field_days_first_visit_proxy': burden_first_visit['scorable_field_days'],
        })
    for baseline_name, baseline in cohort['development_fitted_calendar_baselines'].items():
        calendar_rows.append({
            'mode': mode,
            'baseline': baseline_name,
            'window': f"{baseline['start_month_day']} — {baseline['end_month_day']}",
            'n': baseline['n'],
            'hits': baseline['hits'],
            'hit_rate': baseline['hit_rate'],
            'june_august_alarm_day_fraction': baseline['june_august_alarm_day_fraction'],
            'pre_detection_alarm_fraction_june1': baseline['pre_detection_alarm_day_fraction_june1'],
            'pre_detection_alarm_fraction_first_visit_proxy': baseline['pre_detection_alarm_day_fraction_first_visit_proxy'],
        })
    primary = summary['polyakov'][mode][COHORT]['observed_bbch51_operational_point_primary']
    endpoint = primary['endpoints']['activation_any__onset_any']
    paired = endpoint['paired_vs_phenology_only']
    permutation = primary['within_season_date_permutation_any_overlap']
    polyakov_burden = primary['operational_alarm_burden_before_detection']
    polyakov_rows.append({
        'mode': mode,
        'endpoint': 'activation_any__onset_any',
        'n_paired': paired['n_paired'],
        'polyakov_hits': endpoint['polyakov']['successes'],
        'polyakov_rate': paired['model_rate'],
        'phenology_only_hits': endpoint['phenology_only']['successes'],
        'phenology_only_rate': paired['baseline_rate'],
        'paired_lift': paired['absolute_lift'],
        'mcnemar_p_two_sided': paired['mcnemar_p_two_sided'],
        'within_season_null': permutation['null_mean_rate'],
        'permutation_p_one_sided': permutation['permutation_p_one_sided'],
        'at_risk_field_days': polyakov_burden['at_risk_field_days'],
        'polyakov_actionable_alarm_days': polyakov_burden['polyakov_actionable_alarm_days'],
        'polyakov_actionable_alarm_fraction': polyakov_burden['polyakov_actionable_alarm_fraction'],
        'phenology_only_alarm_days': polyakov_burden['phenology_only_alarm_days'],
        'phenology_only_alarm_fraction': polyakov_burden['phenology_only_alarm_fraction'],
    })

def formatted(frame, percent_columns=(), p_columns=()):
    result = pd.DataFrame(frame).copy()
    for column in percent_columns:
        result[column] = result[column].map(lambda x: '—' if pd.isna(x) else f'{x:.1%}')
    for column in p_columns:
        result[column] = result[column].map(lambda x: '—' if pd.isna(x) else f'{x:.3f}')
    return result

print('Hutton: temporal test slice 2023–2025, только поля вне development')
display(formatted(hutton_rows,
    ('hit_rate', 'within_season_null', 'lift_vs_null', 'pre_detection_alarm_fraction_june1', 'pre_detection_alarm_fraction_first_visit_proxy'),
    ('permutation_p_one_sided',)))
print('Календарные baseline-окна: fitted на development и фиксированный июль–август')
display(formatted(calendar_rows,
    ('hit_rate', 'june_august_alarm_day_fraction', 'pre_detection_alarm_fraction_june1', 'pre_detection_alarm_fraction_first_visit_proxy')))
print('Поляков: BBCH 51; phenology-only baseline = фиксированное BBCH51+15…17 без погодного фильтра')
display(formatted(polyakov_rows,
    ('polyakov_rate', 'phenology_only_rate', 'paired_lift', 'within_season_null', 'polyakov_actionable_alarm_fraction', 'phenology_only_alarm_fraction'),
    ('mcnemar_p_two_sided', 'permutation_p_one_sided')))

In [ ]:
h = summary['hutton']['direct'][COHORT]
p = summary['polyakov']['direct'][COHORT]['observed_bbch51_operational_point_primary']
endpoint = p['endpoints']['activation_any__onset_any']
paired = endpoint['paired_vs_phenology_only']
negative = summary['negative_only_season_sensitivity_not_specificity']['direct'][COHORT]
print('ИТОГ')
for lookback in (7, 14, 21):
    metric = h[f'lookback_{lookback}d']
    observed = metric['first_detection']
    null = metric['within_season_date_permutation_null']
    burden_june1 = metric['pre_detection_alarm_burden_june1']
    burden_first_visit = metric['pre_detection_alarm_burden_first_visit_proxy']
    print(f"Hutton {lookback}d: {observed['successes']}/{observed['n']} попаданий; "
          f"внутрисезонный null={null['null_mean_rate']:.1%}; "
          f"p={null['permutation_p_one_sided']:.3f}; "
          f"тревожные дни 1 июня→обнаружение={burden_june1['alarm_day_fraction']:.1%}; "
          f"first-visit proxy={burden_first_visit['alarm_day_fraction']:.1%}.")
print(f"Поляков / activation_any__onset_any: "
      f"{endpoint['polyakov']['successes']}/{endpoint['polyakov']['n']} против "
      f"{endpoint['phenology_only']['successes']}/{endpoint['phenology_only']['n']} у phenology-only; "
      f"парный lift={paired['absolute_lift']:.1%} при n={paired['n_paired']}.")
polyakov_burden = p['operational_alarm_burden_before_detection']
print(f"Actionable burden до detection: "
      f"{polyakov_burden['polyakov_actionable_alarm_days']}/"
      f"{polyakov_burden['at_risk_field_days']} field-days у Полякова против "
      f"{polyakov_burden['phenology_only_alarm_days']}/"
      f"{polyakov_burden['at_risk_field_days']} у phenology-only; день выпуска и detection исключены.")
print('Эти результаты проверяют временную ассоциацию с первым зарегистрированным обнаружением,')
print('но сами по себе не доказывают биологическую чувствительность, specificity или пользу live-прогноза.')
print()
print('ОГРАНИЧЕНИЯ ОТРИЦАТЕЛЬНЫХ СЕЗОНОВ')
print(f"Всего negative-only сезонов: {negative['all_negative_only_seasons']}; "
      f"с плотным позднесезонным наблюдением: {negative['dense_late_season_followup_seasons']}; "
      f"с полным покрытием риска: {negative['strict_full_risk_season_coverage_seasons']}; "
      f"со строгим покрытием и явным отрицательным свидетельством: "
      f"{negative['strict_explicit_coverage_seasons']}.")
print('Это только анализ чувствительности к определению отрицательного сезона, не оценка true negative/specificity:')
print('в данных нет гарантированного регулярного полного осмотра, сведений обо всех обработках и подтверждения,')
print('что болезнь действительно отсутствовала до конца периода риска.')

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

# Публичный архив содержит только агрегаты. Event-level CSV и weather_metadata
# остаются локально: в них есть идентификаторы и/или точные геопривязки полей.
aggregate_files = [OUTPUT_DIR / 'summary.json']
report_path = OUTPUT_DIR / 'report_ru.md'
if report_path.exists():
    aggregate_files.append(report_path)
for path in aggregate_files:
    if not path.exists():
        raise FileNotFoundError(path)
archive = OUTPUT_DIR.parent / 'vaad_late_blight_aggregate.zip'
with ZipFile(archive, 'w', compression=ZIP_DEFLATED) as bundle:
    for path in aggregate_files:
        bundle.write(path, arcname=path.name)
print('Aggregate-only результаты:', archive)
print('Состав архива:', [path.name for path in aggregate_files])
if IN_COLAB:
    from google.colab import files
    files.download(str(archive))